# T-10 — MODIS–VIIRS overlap analysis and calibration period

Goal: choose the calibration period for the VIIRS → MODIS-equivalent conversion (FR-D-09).

Inputs: `data/processed/grid_monthly.parquet` (T-09) and, if present, `data/processed/cells.parquet` (T-03) for biome regions.
Run the pipeline first: `make data` (or `firecal ingest && firecal clean && firecal grid`).

The ratio shown is **MODIS / (VIIRS per active satellite)**, so it is not inflated when NOAA-20 (2018) and NOAA-21 (2024) join S-NPP.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from firecal.config import get_settings
from firecal.overlap import monthly_pairs, suggest_period, yearly_ratios

# Notebooks run from pipeline/notebooks; point at the repo data folder.
if not Path(os.environ.get("DATA_ROOT", "data")).exists():
    os.chdir(Path.cwd().parents[1])
settings = get_settings()
pairs = monthly_pairs(settings.paths, settings.grid)
ratios = yearly_ratios(pairs)
ratios.pivot(index="year", columns="region", values="ratio").round(3)

## Ratio per year and region

A stable calibration period shows flat lines. Watch for: the 2012 S-NPP ramp-up, years with exceptional fire seasons in one region, and the post-2021 Terra/Aqua orbit drift (later equator crossing → different diurnal sampling).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for region, g in ratios[ratios.months == 12].groupby("region"):
    ax.plot(g.year, g.ratio, marker="o", lw=2.5 if region == "global" else 1, label=region)
ax.set(xlabel="year", ylabel="MODIS / VIIRS per satellite", title="Yearly detection ratio")
ax.legend(fontsize=8, ncols=2)
ax.grid(alpha=0.3)

In [ ]:
glob = pairs.groupby(["year", "month"])[["n_modis", "v_norm"]].sum().reset_index()
glob["t"] = glob.year + (glob.month - 0.5) / 12
glob["ratio"] = glob.n_modis / glob.v_norm
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(glob.t, glob.ratio, lw=0.8)
ax.plot(glob.t, glob.ratio.rolling(12, center=True).median(), lw=2, label="12-month rolling median")
ax.set(xlabel="year", ylabel="ratio", title="Monthly global ratio (seasonality vs drift)")
ax.legend()
ax.grid(alpha=0.3)

## Suggested period

`suggest_period` keeps the longest contiguous run of full years (after the VIIRS start year, up to the drift year) whose global ratio stays within ±15 % of the median.

In [ ]:
choice = suggest_period(ratios, tolerance=0.15, drift_year=2021)
print(
    "Suggested CALIBRATION_PERIOD =", choice.period, "| configured =", settings.calibration_period
)
choice.table[["year", "months", "ratio", "deviation", "eligible", "stable"]].round(3)

## Decision

Recorded in [`docs/calibration-period.md`](../../docs/calibration-period.md). Update that file and `CALIBRATION_PERIOD` in `.env` if the analysis on the full FIRMS archive suggests a different window.